# Learning the Finite Element Method — Step 4 (verification)

This final, short step checks the implementation rather than adding new FEM machinery. It reuses the definitions from Step 3, then verifies reference matrices, boundary algebra, an affine patch, and the expected $L^2$ convergence rate.


In [1]:
%%capture
%run ./Learning_FEM_1D_step3.ipynb


## Reference element and boundary algebra

For P1 functions on $[-1,1]$, the three reference arrays are known exactly. We also check the outward-current signs on a two-node system.


In [2]:
solver = FEM_solver(n_q=2)

assert np.allclose(solver.Kxx, [[0.5, -0.5], [-0.5, 0.5]])
assert np.allclose(solver.M, [[2/3, 1/3], [1/3, 2/3]])
assert np.allclose(solver.Q, [1.0, 1.0])

A_test = sparse.lil_matrix([[2.0, -1.0], [-1.0, 2.0]])
b_test = np.zeros(2)
bc_test = {
    "xmin": {"type": "neumann", "value": -3.0},
    "xmax": {"type": "robin", "alpha": 0.5, "beta": 2.0},
}
A_test, b_test = solver.apply_boundary_conditions(A_test, b_test, bc_test, 2)
assert np.allclose(A_test.toarray(), [[2.0, -1.0], [-1.0, 2.5]])
assert np.allclose(b_test, [3.0, 2.0])
print("Reference arrays and boundary signs: passed")


Reference arrays and boundary signs: passed


## Affine patch and symmetry

With zero reaction and source, an affine function is reproduced exactly. Symmetric elimination should also leave the assembled matrix symmetric.


In [3]:
mesh = MESH([0], [0], [1.0], [8])
prop = {"D": np.array([1.0]), "siga": np.array([0.0]), "src": np.array([0.0])}
bc = {
    "xmin": {"type": "dirichlet", "value": 2.0},
    "xmax": {"type": "dirichlet", "value": 5.0},
}
A, b = solver.assemble_system(mesh, prop, bc)
u = solver.solve_system(A, b)

assert np.allclose(A.toarray(), A.toarray().T)
assert np.allclose(u, 2.0 + 3.0 * mesh.x)
print("Affine patch and symmetry: passed")


Affine patch and symmetry: passed


## Convergence against an analytical solution

For $-D u''=q$ on $[0,L]$ with homogeneous Dirichlet data, $u(x)=q\,x(L-x)/(2D)$. The P1 $L^2$ error should decrease approximately as $h^2$.


In [4]:
def l2_error(mesh, u_h, exact):
    xi, weight = np.polynomial.legendre.leggauss(3)
    error_sq = 0.0
    for e, nodes in enumerate(mesh.gn):
        xq = 0.5 * (mesh.x[e] + mesh.x[e + 1]) + mesh.J[e] * xi
        uq = 0.5 * (1.0 - xi) * u_h[nodes[0]] + 0.5 * (1.0 + xi) * u_h[nodes[1]]
        error_sq += mesh.J[e] * np.sum(weight * (uq - exact(xq))**2)
    return np.sqrt(error_sq)

D, q, L = 1.5, 7.0, 1.0
exact = lambda x: q * x * (L - x) / (2.0 * D)
errors = []
for n_elem in (8, 16, 32):
    mesh = MESH([0], [0], [L], [n_elem])
    prop = {"D": np.array([D]), "siga": np.array([0.0]), "src": np.array([q])}
    bc = {
        "xmin": {"type": "dirichlet", "value": 0.0},
        "xmax": {"type": "dirichlet", "value": 0.0},
    }
    A, b = solver.assemble_system(mesh, prop, bc)
    errors.append(l2_error(mesh, solver.solve_system(A, b), exact))

rates = np.log2(np.asarray(errors[:-1]) / np.asarray(errors[1:]))
print("L2 errors:", np.asarray(errors))
print("observed rates:", rates)
assert np.all(rates > 1.9)


L2 errors: [0.00665635 0.00166409 0.00041602]
observed rates: [2. 2.]
